# 案例二：安居客二手房信息爬取

爬取安居客二手房信息，将房源标题、楼盘、地址、房屋户型、建筑面积、所在楼层、建造年代和每平方米单价保存到 MySQL 数据库。

## 一、主要知识点

- Requests 发送 GET 请求获取网页 HTML。
- XPath 定位房源信息并提取文本内容。
- 通过“下一页”链接实现分页爬取。
- PyMySQL 连接 MySQL，使用 `executemany()` 批量插入数据。

In [ ]:
%pip install requests lxml pymysql

In [1]:
import requests
from lxml import etree
import pymysql

## 二、创建数据库和数据表

在 `spider` 数据库中创建 `houseinfo` 表，字段设置与教材表 5-5 一致。

In [2]:
dbinfo = {
    "host": "127.0.0.1",
    "port": 3306,
    "user": "root",
    "password": "123456",
    "charset": "utf8mb4",
    "cursorclass": pymysql.cursors.DictCursor
}

# 创建数据库
conn = pymysql.connect(
    host=dbinfo["host"],
    port=dbinfo["port"],
    user=dbinfo["user"],
    password=dbinfo["password"],
    charset=dbinfo["charset"]
)
cursor = conn.cursor()
cursor.execute("CREATE DATABASE IF NOT EXISTS spider DEFAULT CHARACTER SET utf8mb4")
conn.commit()
cursor.close()
conn.close()

dbinfo["database"] = "spider"

# 创建二手房信息表
conn = pymysql.connect(**dbinfo)
cursor = conn.cursor()
cursor.execute("""
CREATE TABLE IF NOT EXISTS houseinfo (
    id INT NOT NULL AUTO_INCREMENT PRIMARY KEY,
    title VARCHAR(200) NOT NULL,
    house VARCHAR(50),
    address VARCHAR(200),
    struct VARCHAR(50),
    area VARCHAR(20),
    floor VARCHAR(20),
    maketime VARCHAR(20),
    price VARCHAR(20)
) DEFAULT CHARSET=utf8mb4
""")
conn.commit()
cursor.close()
conn.close()

## 三、获取网页数据

使用 Requests 发送 GET 请求，并设置 User-Agent 请求头。

In [3]:
def get_html(url, time=30):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/80.0.3987.132 Safari/537.36"
    }
    try:
        r = requests.get(url, headers=headers, timeout=time)
        r.encoding = r.apparent_encoding
        r.raise_for_status()
        return r.text
    except Exception as error:
        print(error)
        return None

## 四、观察网页结构

房源信息主要位于 `div.property` 标签中。先查看一条房源的 HTML 结构，再根据标签和属性确定 XPath 表达式。

In [4]:
url = "https://www.bspider.top/anjuke/"
html = get_html(url)
doc = etree.HTML(html)
rows = doc.xpath("//div[@class='property']")

print("房源节点数量：", len(rows))
if rows:
    print(etree.tostring(rows[0], encoding="utf-8").decode("utf-8"))
else:
    print(html[:1500])

房源节点数量： 60
<div tongji_tag="fcpc_ersflist_gzcount" class="property" data-v-53a259aa="" data-v-31e248e6=""><a href="https://chongqing.anjuke.com/prop/view/A5887814286?auction=201&amp;hpType=1&amp;entry=102&amp;position=0&amp;kwtype=filter&amp;now_time=1628513553&amp;spread=filtersearch_p&amp;from=from_esf_List_search&amp;index=0" data-action="esf_list" target="_blank" data-ep="{&quot;exposure&quot;:{&quot;esf_list&quot;:1,&quot;broker_id&quot;:&quot;101348367&quot;}}" data-cp="{&quot;broker_id&quot;:&quot;101348367&quot;}" data-lego="{&quot;entity_id&quot;:&quot;2115457344527366&quot;,&quot;tid&quot;:&quot;-&quot;}" class="property-ex" data-v-53a259aa=""><div class="property-image" data-v-53a259aa=""><img alt="" src="/static/anjuke/images/120.jpg" class="lazy-img cover" data-v-28819723="" data-v-53a259aa=""/> <img src="/static/anjuke/images/comm_propdetail_icon_vr_m@2x.png" class="property-image-vr" data-v-53a259aa=""/> <span id="main-0" class="property-image-vr" data-v-53a259aa=""/></d

## 五、解析房源信息

遍历每个 `div.property` 节点，提取标题、楼盘名称、地址、户型、面积、楼层、建造年代和单价。房屋基本信息通过 `string()` 获取后，去除多余空格和换行，再按顺序拆分。

In [5]:
out_list = []

def parser(html):
    doc = etree.HTML(html)

    for row in doc.xpath("//div[@class='property']"):
        # 房源标题
        title = row.xpath(".//h3/@title")
        if title:
            title = title[0].strip()
        else:
            title = "".join(row.xpath(".//h3//text()")).strip()

        # 房型、面积、楼层和建造年代
        info = row.xpath("string(.//*[@class='property-content-info'])")
        info = info.replace(" ", "").replace("\n\n", "\n").split("\n")

        # 楼盘名称和地址
        house = row.xpath(".//*[@class='property-content-info-comm-name']/text()")
        house = house[0].strip() if house else ""
        address = row.xpath("string(.//*[@class='property-content-info-comm-address'])")

        # 每平方米单价
        price = row.xpath(".//*[@class='property-price-average']/text()")
        price = price[0].replace(" ", "").strip() if price else ""

        row_list = [
            title,
            house,
            address.strip(),
            info[0] if len(info) >= 1 else "",
            info[1] if len(info) >= 2 else "",
            info[3] if len(info) >= 4 else "",
            info[4] if len(info) >= 5 else "",
            price
        ]
        out_list.append(row_list)

    # 判断是否存在下一页
    ele_next = doc.xpath("//a[contains(@class,'next-active')]/@href")
    if ele_next:
        new_url = ele_next[0]
        if new_url.startswith("?page="):
            new_url = "https://www.bspider.top/anjuke/" + new_url
        next_html = get_html(new_url)
        if next_html:
            parser(next_html)

    return out_list

## 六、保存到 MySQL

将解析得到的列表传入 `executemany()`，批量写入 `houseinfo` 表。

In [6]:
def save_mysql(sql, val, **dbinfo):
    try:
        connect = pymysql.connect(**dbinfo)
        cursor = connect.cursor()
        cursor.executemany(sql, val)
        connect.commit()
    except Exception as error:
        connect.rollback()
        print(error)
    finally:
        cursor.close()
        connect.close()

## 七、运行程序

In [7]:
out_list = []
url = "https://www.bspider.top/anjuke/"
html = get_html(url)
parser(html)

print("共解析房源：", len(out_list), "条")

sql = "INSERT INTO houseinfo(title,house,address,struct,area,floor,maketime,price) VALUES(%s,%s,%s,%s,%s,%s,%s,%s)"
save_mysql(sql, out_list, **dbinfo)

共解析房源： 600 条


## 八、查看存储结果

In [8]:
conn = pymysql.connect(**dbinfo)
cursor = conn.cursor()
cursor.execute("SELECT * FROM houseinfo ORDER BY id DESC LIMIT 5")
for item in cursor.fetchall():
    print(item)
cursor.close()
conn.close()

{'id': 600, 'title': '恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售', 'house': '恒大照母山山水城', 'address': '渝北照母山恒康路9号', 'struct': '3室2厅1卫', 'area': '90.2㎡', 'floor': '低层(共32层)', 'maketime': '2015年建造', 'price': '15530元/㎡'}
{'id': 599, 'title': '电梯房 恒大照母山山水城 90平 满二 143万 看房方便', 'house': '恒大照母山山水城', 'address': '渝北照母山恒康路9号', 'struct': '3室2厅1卫', 'area': '90.2㎡', 'floor': '低层(共32层)', 'maketime': '2016年建造', 'price': '15863元/㎡'}
{'id': 598, 'title': '毛坯 3室2厅 电梯房 首地江山赋 91平 145万', 'house': '首地江山赋', 'address': '渝北悦来悦山路5号', 'struct': '3室2厅1卫', 'area': '91.8㎡', 'floor': '低层(共33层)', 'maketime': '2016年建造', 'price': '15799元/㎡'}
{'id': 597, 'title': '急售 照母山 不到150万的三房 精装修 随时看', 'house': '恒大照母山山水城', 'address': '渝北照母山恒康路9号', 'struct': '3室2厅1卫', 'area': '90.2㎡', 'floor': '低层(共32层)', 'maketime': '2012年建造', 'price': '15863元/㎡'}
{'id': 596, 'title': '毛坯 3室2厅 145万 电梯房 91平', 'house': '首地江山赋', 'address': '渝北悦来悦山路5号', 'struct': '3室2厅1卫', 'area': '91.8㎡', 'floor': '低层(共33层)', 'maketime': '2016年建造', 'price': '15799元/㎡'}
